# Comparing Metaflow @batch vs @kubernetes vs local execution for cost-performance tradeoffs

## Purpose

This notebook documents the three execution backends Metaflow supports — local, AWS Batch (`@batch`), and Kubernetes (`@kubernetes`) — and provides a framework for comparing their cost and performance characteristics. The same `FlowSpec` can run any step on any backend without restructuring the flow, making it straightforward to evaluate tradeoffs per step.

## When to use

| Backend | Best suited for | Avoid when |
|---|---|---|
| local | Development, debugging, small datasets, fast iteration | Workloads exceeding laptop resources; need for reproducibility across environments |
| `@batch` (AWS) | Bursty or long-running CPU/GPU jobs; no cluster to manage | Latency-sensitive steps where seconds of spin-up matter; environments without AWS Batch access |
| `@kubernetes` | Steady workloads on an existing cluster; shared GPU scheduling; multi-team environments | No cluster available; cluster capacity or quota constraints prevent scheduling |

The guiding principle: keep steps under an hour and only annotate compute-intensive steps. Spawning a Batch job for a 2-second step costs more in spin-up than in compute.

## Prerequisites

- Metaflow installed (`pip install metaflow`)
- AWS Batch environment configured (for `@batch` steps)
- Kubernetes cluster with Metaflow operator installed (for `@kubernetes` steps)
- Python 3.9+ and the `metaflow` package

## Steps

Define a single flow with three parallel branches, each targeting a different backend. The `start` step fans out to three training steps, each annotated with a different decorator.

In [ ]:
from metaflow import FlowSpec, step, Parameter, resources

class BackendCompareFlow(FlowSpec):

    epochs = Parameter("epochs", help="Training epochs", default=5)

    @step
    def start(self):
        self.config = {"lr": 0.01, "epochs": self.epochs}
        print("start: preparing config (local)")
        self.next(self.train_local, self.train_batch, self.train_k8s)

    @step
    def train_local(self):
        import time
        time.sleep(1)
        self.backend = "local"
        self.result = sum(range(1000)) / 1000.0
        print(f"train_local: computed on {self.backend}")
        self.next(self.join)

    @resources(cpu=2, memory=4096)
    @step
    def train_batch(self):
        import time
        time.sleep(1)
        self.backend = "aws_batch"
        self.result = sum(range(1000)) / 1000.0
        print(f"train_batch: computed on {self.backend}")
        self.next(self.join)

    @resources(cpu=2, memory=4096)
    @step
    def train_k8s(self):
        import time
        time.sleep(1)
        self.backend = "kubernetes"
        self.result = sum(range(1000)) / 1000.0
        print(f"train_k8s: computed on {self.backend}")
        self.next(self.join)

    @step
    def join(self, inputs):
        self.backends = [i.backend for i in inputs]
        self.results = [i.result for i in inputs]
        print(f"join: merged backends={self.backends} results={self.results}")

if __name__ == "__main__":
    BackendCompareFlow()

### Running the flow

Run the flow locally first to validate the DAG logic. The `@batch` and `@kubernetes` steps will execute locally without cloud infrastructure configured.

In [ ]:
# Run the full flow locally; all three backends execute on the local machine.
python backend_compare_flow.py run

To push specific steps to cloud backends, use the `--with` flag:

In [ ]:
# Push the batch and kubernetes steps to their respective backends.
python backend_compare_flow.py run --with batch --with kubernetes

## Verify

After a cloud-enabled run, list the runs and inspect the artifacts to confirm each branch executed on its intended backend.

In [ ]:
python backend_compare_flow.py list

In [ ]:
from metaflow import Flow

run = Flow("BackendCompareFlow").latest_successful_run
print("backends:", run.data.backends)
print("results:", run.data.results)

## Common errors

- `@batch` and `@kubernetes` decorators are inert without `--with batch` / `--with kubernetes`, or a configured cloud environment. Without these flags, all steps run locally.
- `@resources(cpu=, memory=)` is a hint to the scheduler; AWS Batch maps it to a job definition and Kubernetes maps it to pod resource requests. Mismatched requests can cause steps to sit in pending state if the cluster has no fitting node.
- Cloud steps require the same dependencies available at runtime. When using `@pypi` or `@conda`, pin metaflow-compatible packages so the step's container can import them.
- Artifacts flow back to the central metadata store automatically, but only after the step finishes successfully. A crashed cloud step leaves no artifact for the join step to merge.